# 02 — Prophet Model
**Niloo**

In [7]:
# Imports: all libraries needed for data loading, modelling, plotting and saving
import pandas as pd
import numpy as np
from prophet import Prophet
import joblib
import plotly.graph_objects as go
import os

In [8]:
# Load data: use real SMHI data if available, otherwise fall back to a synthetic sine-wave placeholder
DATA_PATH = os.path.join('..', 'data', 'clean_data.csv')

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH, parse_dates=['ds'])
    print(f'Loaded real data: {len(df)} rows')
else:
    # Fake SMHI-like data: seasonal sine wave between -5 °C and 25 °C
    dates = pd.date_range('1996-01-01', periods=360, freq='ME')
    temps = 10 + 15 * np.sin(2 * np.pi * (np.arange(360) - 3) / 12) + np.random.default_rng(42).normal(0, 1, 360)
    df = pd.DataFrame({'ds': dates, 'y': temps})
    print('clean_data.csv not found — using synthetic placeholder data')

print(df.head())

clean_data.csv not found — using synthetic placeholder data
          ds          y
0 1996-01-31  -4.695283
1 1996-02-29  -4.030365
2 1996-03-31   3.250451
3 1996-04-30  10.940565
4 1996-05-31  15.548965


In [9]:
# Train Prophet model with yearly seasonality and 95 % confidence intervals
model = Prophet(
    interval_width=0.95,
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
)
model.fit(df)
print('Model training complete.')

Model training complete.


In [10]:
# Generate forecast: extend 1 461 days (~4 years) beyond the last historical date
future = model.make_future_dataframe(periods=1461, freq='D')
forecast = model.predict(future)
print(f'Forecast horizon: {forecast["ds"].iloc[-1].date()}')

Forecast horizon: 2029-12-31


In [11]:
# Print the forecasted yearly average temperature for each of the 4 future years
last_historical = df['ds'].max()
future_fc = forecast[forecast['ds'] > last_historical].copy()
future_fc['year'] = future_fc['ds'].dt.year

print('Forecasted yearly average temperatures (°C):')
for year, grp in future_fc.groupby('year'):
    print(f'  {year}: {grp["yhat"].mean():.2f} °C')

Forecasted yearly average temperatures (°C):
  2026: 10.12 °C
  2027: 10.13 °C
  2028: 10.10 °C
  2029: 10.14 °C


In [12]:
# Interactive Plotly chart: historical data, forecast line and 95 % confidence band
fig = go.Figure()

# Confidence interval shaded area
fig.add_trace(go.Scatter(
    x=pd.concat([forecast['ds'], forecast['ds'][::-1]]),
    y=pd.concat([forecast['yhat_upper'], forecast['yhat_lower'][::-1]]),
    fill='toself',
    fillcolor='rgba(99,110,250,0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='95 % CI',
    hoverinfo='skip',
))

# Forecast line
fig.add_trace(go.Scatter(
    x=forecast['ds'],
    y=forecast['yhat'],
    mode='lines',
    line=dict(color='rgb(99,110,250)', width=2),
    name='Forecast',
))

# Historical observations
fig.add_trace(go.Scatter(
    x=df['ds'],
    y=df['y'],
    mode='markers',
    marker=dict(color='rgb(239,85,59)', size=4, opacity=0.7),
    name='Historical',
))

fig.update_layout(
    title='NordCast — Temperature Forecast (Prophet)',
    xaxis_title='Date',
    yaxis_title='Temperature (°C)',
    hovermode='x unified',
    template='plotly_white',
    legend=dict(orientation='h', y=-0.15),
)

fig.show()

In [ ]:
# Save the trained Prophet model to disk so other notebooks can load it without re-training
MODEL_PATH = os.path.join('..', 'models', 'prophet_model.pkl')
os.makedirs(os.path.dirname(MODEL_PATH), exist_ok=True)
joblib.dump(model, MODEL_PATH)
print(f'Model saved to {MODEL_PATH}')